# Sionna RT — v7 multicell trace on Colab free GPU
Re-runs the ray-tracing behind **5G NR Multicell Twin v7** (3 towers x 3 sectors, 3.6 GHz, 25x25 grid at 40 m, street 1.5 m, Vexcel SF mesh) with **Sionna RT 2.2.0** — the same version and script (`rt_multicell.py`) that produced v7, switched from the CPU (LLVM) variant to the GPU (CUDA) variant.

**Before running:** put the scene folder in Google Drive as `MyDrive/sf-twin/scene_sf/` — it must contain `scene.xml` (plus its mesh files) and `height_rasters.npz`. Everything else (the v7 code) is pulled from GitHub automatically.

**Run:** Runtime -> Change runtime type -> **T4 GPU**, then Runtime -> Run all.

**Disconnect-safe:** after every chunk the results file is backed up to `MyDrive/sionna-v7-out/`. If the free session dies, reconnect and Run all again — finished chunks are restored from Drive and skipped.

**Check at the end:** the post step should print coverage (RSRP >= -90 dBm) about **80.5%** and median best-server RSRP about **-72.7 dBm** — the published v7 numbers.

In [ ]:
# 1. GPU check
!nvidia-smi
import os
assert os.path.exists('/proc/driver/nvidia/version'), 'No GPU runtime: Runtime -> Change runtime type -> T4 GPU, then Run all again.'
print('GPU runtime OK')

In [ ]:
# 2. Install Sionna RT 2.2.0 (the version v7 was traced with)
!pip install -q "sionna-rt==2.2.0"
import importlib.metadata as md_
for p in ('sionna-rt', 'mitsuba', 'drjit', 'numpy'):
    try: print(p, md_.version(p))
    except Exception as e: print(p, '??', e)

In [ ]:
# 3. Mount Drive and find the scene
from google.colab import drive
drive.mount('/content/drive')
import os
SCENE = '/content/drive/MyDrive/sf-twin/scene_sf'
for f in ('scene.xml', 'height_rasters.npz'):
    p = os.path.join(SCENE, f)
    print(f, 'OK', os.path.getsize(p) if os.path.exists(p) else 'MISSING')
assert os.path.exists(os.path.join(SCENE, 'scene.xml')), 'Scene not in Drive: upload the scene_sf folder to MyDrive/sf-twin/ first.'
print('Scene found:', SCENE)

In [ ]:
# 4. Fetch the v7 code from GitHub and wire up the paths the scripts expect
import os, shutil, subprocess
WORK = '/content/mcsf-work'
if os.path.exists(WORK): shutil.rmtree(WORK)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ysrao/twin5GNR-multicell-SF.git', '/content/mcsf'], check=True)
subprocess.run(['unzip', '-q', '-o', '/content/mcsf/multicell-code-v7-v8.zip', '-d', WORK], check=True)
# rt_multicell.py hardcodes /home/hatch/workspace/sf-twin ; compare_veg.py and build_v7.py
# hardcode /home/hatch/workspace/twin5gnr-work/twin5GNR-multicell-sionna . Colab runs as
# root, so satisfy both with symlinks instead of editing the scripts.
os.makedirs('/home/hatch/workspace/twin5gnr-work', exist_ok=True)
def link(dst, src):
    if os.path.islink(dst) or os.path.exists(dst): os.remove(dst) if os.path.islink(dst) or os.path.isfile(dst) else shutil.rmtree(dst)
    os.symlink(src, dst)
link('/home/hatch/workspace/sf-twin', '/content/drive/MyDrive/sf-twin')
link('/home/hatch/workspace/twin5gnr-work/twin5GNR-multicell-sionna', WORK)
print('code at', WORK, '| files:', len(os.listdir(WORK)))

In [ ]:
# 5. Switch the tracer to the GPU (CUDA) variant, with automatic CPU fallback
import re
p = os.path.join(WORK, 'rt_multicell.py')
s = open(p).read()
assert 'llvm_ad_mono_polarized' in s
open(p, 'w').write(s.replace('llvm_ad_mono_polarized', 'cuda_ad_mono_polarized'))
try:
    import mitsuba as mi
    mi.set_variant('cuda_ad_mono_polarized')
    import drjit as dr
    print('CUDA variant OK, devices:', dr.cuda.device_count() if hasattr(dr, 'cuda') else '?')
except Exception as e:
    print('CUDA variant failed (', e, ') - falling back to CPU/LLVM variant (same results, slower)')
    open(p, 'w').write(open(p).read().replace('cuda_ad_mono_polarized', 'llvm_ad_mono_polarized'))

In [ ]:
# 6. Restore any backed-up progress from Drive (free sessions disconnect)
import os, shutil
OUT = '/content/drive/MyDrive/sionna-v7-out'
os.makedirs(OUT, exist_ok=True)
NPZ = os.path.join(WORK, 'results_multicell_f3600_street.npz')
bak = os.path.join(OUT, 'results_multicell_f3600_street.npz')
if os.path.exists(bak) and not os.path.exists(NPZ):
    shutil.copy(bak, NPZ); print('restored backup from Drive')
elif os.path.exists(bak):
    print('backup exists in Drive; local copy kept')
else:
    print('no backup yet - fresh run')

In [ ]:
# 7. Run the trace: 5 chunks (625 points, 150 per chunk), backing up after each
import subprocess, shutil, os
for i in range(5):
    r = subprocess.run(['python3', 'rt_multicell.py', '--chunk', str(i)], cwd=WORK, capture_output=True, text=True)
    print(r.stdout.strip()[-300:])
    if r.returncode != 0:
        print(r.stderr[-2000:]); raise SystemExit(f'chunk {i} failed')
    shutil.copy(NPZ, bak)  # backup after every chunk
print('all chunks done')

In [ ]:
# 8. Post-process and validate against the published v7 numbers
import subprocess, os
r = subprocess.run(['python3', 'rt_multicell.py', '--post'], cwd=WORK, capture_output=True, text=True)
print(r.stdout); print(r.stderr[-1000:] if r.returncode else '')
assert r.returncode == 0
print('Expected (published v7): coverage ~80.5%, median best-server RSRP ~-72.7 dBm')

In [ ]:
# 9. Vegetation layer (P.833 surveyed trees, already in the code package) + rebuild the v7 page
import subprocess, shutil, os
for script in ('compare_veg.py', 'build_v7.py'):
    r = subprocess.run(['python3', script], cwd=WORK, capture_output=True, text=True)
    print(script, '->', r.returncode); print((r.stdout or r.stderr)[-500:])
    assert r.returncode == 0, script + ' failed'
for f in ('results_multicell_f3600_street.npz', 'comparison_veg.json', 'results_veg_40m.npz', '6g-ran-multicell-handoff-v7.html'):
    shutil.copy(os.path.join(WORK, f), os.path.join(OUT, f))
print('Outputs saved to MyDrive/sionna-v7-out/: results npz, comparison_veg.json, results_veg_40m.npz, v7 html')

## Notes
- **Why CUDA vs the original run:** v7 was traced with the LLVM (CPU) Mitsuba variant. The CUDA variant computes the same paths; expect agreement to ~0.01 dB, and the step-8 check against the published 80.5% / -72.7 dBm figures confirms it.
- **Free-tier limits:** sessions cap at ~12 h and disconnect after ~90 min idle — this notebook is built for that (per-chunk Drive backups, resumable chunks).
- **To change the scenario** (tower positions, frequency, grid): edit `rt_multicell.py` in `/content/mcsf-work` (TOWERS, STEP, TILTS at the top) before step 7, or ask Muse to prepare a variant notebook.